# Capstone Progect [ALL FUNCTIONS]
## How to approach this file
* Download inputs and outputs for weekly submissions
* create a folder in the following format week\<week_number\>
* move inputs.txt and outputs.txt into folder
* use the load_and_append_inputs_and_outputs_by_week_and_functionId()
  * N.B. the inputs.txt and outputs.txt are loaded each week with the previous' week's submissions. Therefore, the values in the files are appended to the initial files
* enjoy

In [395]:
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import ast
import re

### COMMON TO ALL FUNCTIONS
# create df from inputs, columns
def create_and_display_df(inputs, output, input_columns, output_columns='output'):
    print_shapes(inputs, output)
    df = pd.DataFrame(inputs, columns=input_columns)
    df[output_columns] = output
    display(df)
    return

def print_shapes(inputs, output):
    print(inputs.shape)
    print(output.shape)
    return

def load_data(function_id, week):
    base = f"function{function_id}/week{week}"
    print(base)
    X = np.load(f"{base}/initial_inputs.npy")
    Y = np.load(f"{base}/initial_outputs.npy")
    return X, Y

# UCB(x) = mu(x) + beta * variance/sigma(x)
def acquisition_ucb(mu, sigma, beta=1.96):
    return mu + beta * sigma

def acquisition_ei2(mu, sigma, best_Y, xi):
    # this is the 'true' EI. The official function 
    # https://botorch.org/docs/acquisition/
    improvement = mu - best_Y - xi
    z = improvement / sigma
    ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)
    ei[sigma == 0.0] = 0.0
    return ei 

def acquisition_ei(mu, sigma, best_Y, xi):
    return (mu - best_Y - xi) / sigma

#generic noise chosen
def fit_model(X, y, kernel, noise=1e-6):
    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=noise,
        n_restarts_optimizer=9
    )
    model.fit(X, y)
    return model

#all possible points that could be chosen
def generate_candidates(dim, n=5000, lower_bound=0, upper_bound=1):
    return np.random.uniform(lower_bound, upper_bound, size=(n, dim))

# retrieve data from correct folder / week and append to list
def load_and_append_inputs_and_outputs_by_week_and_functionId(input_list, output_list, week, function_id):
    print("Loading new inputs for function", function_id,", week", week)
    # Load inputs
    with open(f"week{week}/inputs.txt", "r") as f:
        content = f.read()
    array = np.array
    arrays = eval(content)
    new_inputs = [arr.tolist() for arr in arrays]

    with open(f"week{week}/outputs.txt", "r") as f:
        content = f.read()
    arrays = eval(content)
    new_outputs = [arr.tolist() for arr in arrays]
    input_list = np.vstack([input_list, new_inputs[function_id-1]])
    output_list = np.concatenate([output_list, [new_outputs[function_id-1]]])
    return input_list, output_list

def parse_input_content(content):
    # Normalize the content: remove 'array(' and ')' wrappers
    cleaned = re.sub(r'array\(', '', content.strip())
    cleaned = re.sub(r'\)', '', cleaned)
    
    # Safely evaluate the cleaned string as a Python literal
    parsed = ast.literal_eval(cleaned)
    
    # Convert each inner list to a numpy array
    arrays = [np.array(row) for row in parsed]
    
    return arrays

def parse_output_content(content):
    """Parse a line of text into numpy arrays"""
    try:
        # Clean the line and convert to a list of floats
        # This assumes each line contains a list of numbers in string format
        content = content.strip()

        # remove np.float64(...) wrapper
        content = re.sub(r'np\.float64\((.*?)\)', r'\1', content)

        # safely parse the list
        values = ast.literal_eval(content)

        return np.array(values, dtype=float)

    except Exception as e:
        print(f"Error parsing line: {content}")
        print(f"Error details: {e}")
        return np.array([[]])  # Return empty array on error

def load_and_append(input_list, output_list, week, function_id):
    print(f"*** Loading new inputs for week {week}, function {function_id} ***")

    # append new inputs to input_list 
    with open(f"week{week}/inputs.txt", "r") as f:
        arrays = [parse_input_content(line) for line in f if line.strip()]
        final_i = 0
        for i in range(len(arrays)):
            final_i = i
            # append arrays all the weeks results into initial array
            input_list = np.vstack((input_list, arrays[i][function_id - 1]))

    # append new inputs to output_list
    with open(f"week{week}/outputs.txt", "r") as f:
        parsed = [np.array(parse_output_content(line)) for line in f if line.strip()]
        arrays = np.vstack(parsed)
        final_i = 0
        for i in range(len(arrays)):
            final_i = i
            # append arrays all the weeks results into initial array
            output_list =  np.hstack((output_list, arrays[i][function_id - 1]))

    #print(output_list.reshape(-1, 1))
    return input_list, output_list

def print_next_submission(x_next):
    print("-".join(v for v in x_next))
    
class BayesianOptimizer:
    def __init__(self, X, Y, acquisition="ucb", beta=2.0, length_scale=0.2, noise=1e-6, xi=0.01, length_scale_bounds="fixed"):
        self.X = X
        self.Y = Y
        self.acquisition = acquisition
        self.beta = beta
        self.length_scale = length_scale
        self.noise = noise
        self.xi = xi
        self.length_scale_bounds = length_scale_bounds


        kernel = RBF(
            length_scale=self.length_scale,
            length_scale_bounds=self.length_scale_bounds
        )
        self.gp = fit_model(self.X, self.Y, kernel, self.noise)

    def propose(self, n_candidates=5000, lower_bound=0, upper_bound=1, scale=0.2):
        X_cand = ...
        mu=...
        sigma = ...

        if self.acquisition == "ucb":
            X_cand = generate_candidates(self.X.shape[1], n_candidates, lower_bound, upper_bound)
            mu, sigma = self.gp.predict(X_cand, return_std=True)
            scores = acquisition_ucb(mu, sigma, self.beta)
        elif self.acquisition == "ei":
            dim = self.X.shape[1]
            # --- 1. Find current best point ---
            best_idx = np.argmax(self.Y)
            x_best = self.X[best_idx]
            local_candidates = np.random.normal(loc=x_best, scale=scale, size=(n_candidates, dim))
            X_cand = np.clip(local_candidates, lower_bound, upper_bound)
            mu, sigma = self.gp.predict(X_cand, return_std=True)
            best_Y = np.max(self.Y)
            scores = acquisition_ei(mu, sigma, best_Y, xi=self.xi)
        else:
            raise ValueError("Unsupported acquisition")

        x_next = X_cand[np.argmax(scores)]
        return tuple(f"{v:.6f}" for v in x_next)


# Function 1

Detect likely contamination sources in a two-dimensional area, such as a radiation field, where only proximity yields a non-zero reading. The system uses Bayesian optimisation to tune detection parameters and reliably identify both strong and weak sources.

## Week 1

In [293]:
x1, y1 = load_data(1, 1)
columns1 = ['x1', 'x2']
create_and_display_df(x1, y1, columns1)
acquisition1 = "ucb"
beta1 = 1.96
noise1 = 1e-6

bo1 = BayesianOptimizer(x1, y1, acquisition=acquisition1, beta=beta1, noise=noise1, length_scale=100)
next1 = bo1.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next1)

function1/week1
(10, 2)
(10,)


,x1,x2,output
0,0.319404,0.762959,1.322677e-79
1,0.574329,0.879898,1.033078e-46
2,0.731024,0.733000,7.710875e-16
3,0.840353,0.264732,3.341771e-124
4,0.650114,0.681526,-3.606063e-03
5,0.410437,0.147554,-2.159249e-54
6,0.312691,0.078723,-2.089093e-91
7,0.683418,0.861057,2.535001e-40
8,0.082507,0.403488,3.606771e-81
9,0.883890,0.582254,6.229856e-48



The next points to be submitted are: 
0.004778-0.006567


## Week 2

In [294]:

x1w2, y1w2 = load_and_append(x1, y1, 2, 1)

#create_and_display_df(x1w2, y1w2, columns1)
bo1 = BayesianOptimizer(x1w2, y1w2, acquisition=acquisition1, beta=beta1, noise=noise1, length_scale=0.01)
next1 = bo1.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next1)


*** Loading new inputs for week 2, function 1 ***

The next points to be submitted are: 
0.424764-0.441926


It seems like this week, it is still in the exploration phase, giving me various options ranging from the highs and lows

## Week 3

In [295]:
x1w3, y1w3 = load_and_append(x1, y1, 3, 1)
#create_and_display_df(x1w3, y1w3, columns1)
beta1=3
length_scale1=0.2
bo1 = BayesianOptimizer(x1w3, y1w3, acquisition=acquisition1, beta=beta1, noise=noise1, length_scale=length_scale1)
next1 = bo1.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next1)

*** Loading new inputs for week 3, function 1 ***

The next points to be submitted are: 
0.002942-0.971120


## Week 4

In [323]:
x1w4, y1w4 = load_and_append(x1, y1, 4, 1)
#create_and_display_df(x1w4, y1w4, columns1)
beta1=1
length_scale1=0.008
scale1=0.05
n_candidates = 50000
acquisition1 = 'ei'
xi1=0.005
bo1 = BayesianOptimizer(x1w4, y1w4, acquisition=acquisition1, beta=beta1, noise=noise1, length_scale=length_scale1, xi=xi1)
next1 = bo1.propose(n_candidates, scale=length_scale1)
print()
print("The next points to be submitted are: ")
print_next_submission(next1)


*** Loading new inputs for week 4, function 1 ***

The next points to be submitted are: 
0.767559-0.742646


This week, I will change to EI
* it seems as if exploring the 'corner' region as led to a dead center result.
* hypothesis: switching to EI might help exploit point index 4
* outcome: it did not exploit index 4, instead it focused on other corner region. even with an aggressive xi to encourage exploration, that region is filled with uncertainty
* I will submit these points, to better figure out what direction we are heading with this
* i had time to revision it, length scale has be decreased, to keep the candidates closer to the best observation
* xi is smaller still, to improve the best point rather than explore
* candidates have been increased

## Week 5

In [324]:
x1w5, y1w5 = load_and_append(x1, y1, 5, 1)
#create_and_display_df(x1w5, y1w5, columns1)
beta1=1
length_scale1=0.04
scale1=0.03
n_candidates = 50000
acquisition1 = 'ei'
xi1=0.002
bo1 = BayesianOptimizer(
    x1w5, y1w5
    , acquisition=acquisition1, beta=beta1, noise=noise1, length_scale=length_scale1, xi=xi1)
next1 = bo1.propose(n_candidates, scale=scale1)
print()
print("The next points to be submitted are: ")
print_next_submission(next1)


*** Loading new inputs for week 5, function 1 ***

The next points to be submitted are: 
0.676072-0.783532


This week, EI is still the preferred acquisition function
* length scale has been increased slightly
    * implication being that GP slightly connects these points
* scale reduced, reduces exploration
* xi reduced for strong exploitation

# Function 2

Imagine a black box, or a mystery ML model, that takes two numbers as input and returns a log-likelihood score. Your goal is to maximise that score, but each output is noisy, and depending on where you start, you might get stuck in a local optimum.

To tackle this, you use Bayesian optimisation, which selects the next inputs based on what it has learned so far. It balances exploration with exploitation, making it well suited to noisy outputs and complex functions with many local peaks.

## Week 1

In [163]:
x2, y2 = load_data(2, 1)

columns2 = ['x1', 'x2']
create_and_display_df(x2, y2, columns2)
acquisition2 = "ucb"
beta2 = 1.96
noise2 = 1e-6

bo2 = BayesianOptimizer(x2, y2, acquisition=acquisition2, beta=beta2, noise=noise2)
next2 = bo2.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next2)


function2/week1
(10, 2)
(10,)


,x1,x2,output
0,0.665800,0.123969,0.538996
1,0.877791,0.778628,0.420586
2,0.142699,0.349005,-0.065624
3,0.845275,0.711120,0.293993
4,0.454647,0.290455,0.214965
5,0.577713,0.771973,0.023106
6,0.438166,0.685018,0.244619
7,0.341750,0.028698,0.038749
8,0.338648,0.213867,-0.013858
9,0.702637,0.926564,0.611205



The next points to be submitted are: 
0.097771-0.801232


## Week 2

In [164]:
x2w2, y2w2 = load_and_append(x2, y2, 2, 2)
acquisition2='ucb'
bo2 = BayesianOptimizer(x2w2, y2w2, acquisition=acquisition2, beta=beta2, noise=noise2)
next2 = bo2.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next2)


*** Loading new inputs for week 2, function 2 ***

The next points to be submitted are: 
0.976198-0.263395


compared to last week, the only points offered by ucb are 0.99-0.2 tuple
points with ei are 0.7-0.95 ish
both are worth exploring


## Week 3

In [165]:
x2w3, y2w3 = load_and_append(x2, y2, 3, 2)
acquisition2='ucb'
length_scale_bounds2=(0.05, 1.0)
#length_scale_bounds2="fixed"
bo2 = BayesianOptimizer(x2w3, y2w3, acquisition=acquisition2, beta=beta2, noise=noise2, length_scale_bounds=length_scale_bounds2)
next2 = bo2.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next2)

#create_and_display_df(x2w3, y2w3, columns2)


*** Loading new inputs for week 3, function 2 ***

The next points to be submitted are: 
0.928390-0.996398


## Week 4

In [166]:
x2w4, y2w4 = load_and_append(x2, y2, 4, 2)
#create_and_display_df(x2w4, y2w4, columns2)
acquisition2='ucb'
length_scale_bounds2=(0.05, 1.0)
beta2=0.948
#length_scale_bounds2="fixed"
bo2 = BayesianOptimizer(
    x2w4, y2w4
    , acquisition=acquisition2, beta=beta2, noise=noise2, length_scale_bounds=length_scale_bounds2)
next2 = bo2.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next2)



*** Loading new inputs for week 4, function 2 ***

The next points to be submitted are: 
0.460401-0.999795


## Week 5

In [325]:
x2w5, y2w5 = load_and_append(x2, y2, 5, 2)
#create_and_display_df(x2w5, y2w5, columns2)
acquisition2='ei'
length_scale_bounds2=(0.05, 1.0)
beta2=0.7
xi2=0.008
#length_scale_bounds2="fixed"
bo2 = BayesianOptimizer(
    x2w5, y2w5
    , acquisition=acquisition2, beta=beta2, noise=noise2, length_scale_bounds=length_scale_bounds2, xi=xi2)
next2 = bo2.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next2)



*** Loading new inputs for week 5, function 2 ***
(14, 2)
(14,)


,x1,x2,output
0,0.665800,0.123969,0.538996
1,0.877791,0.778628,0.420586
2,0.142699,0.349005,-0.065624
3,0.845275,0.711120,0.293993
4,0.454647,0.290455,0.214965
5,0.577713,0.771973,0.023106
6,0.438166,0.685018,0.244619
7,0.341750,0.028698,0.038749
8,0.338648,0.213867,-0.013858
9,0.702637,0.926564,0.611205



The next points to be submitted are: 
0.716457-0.940265


This week, switch to EI
* xi set to 0.008, strong exploitation to give promising area

# Function 3

You’re working on a drug discovery project, testing combinations of three compounds to create a new medicine.

Each experiment is stored in initial_inputs.npy as a 3D array, where each row lists the amounts of the three compounds used. After each experiment, you record the number of adverse reactions, stored in initial_outputs.npy as a 1D array.

Your goal is to minimise side effects; in this competition, it is framed as maximisation by optimising a transformed output (e.g. the negative of side effects).

## Week 1

In [327]:
x3, y3 = load_data(3, 1)

columns3 = ['x1', 'x2', 'x3']
create_and_display_df(x3, y3, columns3)
acquisition3 = "ei"
beta3 = 1.96
noise3 = 1e-4
xi3 = 0.005

bo3 = BayesianOptimizer(x3, y3, acquisition=acquisition3, beta=beta3, noise=noise3, xi=xi3)
next3 = bo3.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next3)


function3/week1
(15, 3)
(15,)


,x1,x2,x3,output
0,0.171525,0.343917,0.248737,-0.112122
1,0.242114,0.644074,0.272433,-0.087963
2,0.534906,0.398501,0.173389,-0.111415
3,0.492581,0.611593,0.340176,-0.034835
4,0.134622,0.219917,0.458206,-0.048008
5,0.345523,0.941360,0.269363,-0.110621
6,0.151837,0.439991,0.990882,-0.398926
7,0.645503,0.397143,0.919771,-0.113869
8,0.746912,0.284196,0.226300,-0.131461
9,0.170477,0.697032,0.149169,-0.094190



The next points to be submitted are: 
0.336428-0.284636-0.449422


## Week 2

In [328]:
x3w2, y3w2 = load_and_append(x3, y3, 2, 3)
#create_and_display_df(x3w2, y3w2, columns3)

acquisition3='ei'
bo3 = BayesianOptimizer(x3w2, y3w2, acquisition=acquisition3, beta=beta3, noise=noise3, xi=xi3)
next3 = bo3.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next3)


*** Loading new inputs for week 2, function 3 ***

The next points to be submitted are: 
0.326597-0.279687-0.408638


* ei is still better choice here, still sparse coverage, still unknown, need to try to improve with the little attempts i have

## Week 3

In [329]:
x3w3, y3w3 = load_and_append(x3, y3, 3, 3)
#create_and_display_df(x3w3, y3w3, columns3)
xi3=0.05
acquisition3='ei'
n_candidates=20000
lower_bound=0.2
upper_bound=0.8
length_scale_bounds3=(0.05, 2.0) # lets gp adapt smootheness as new data arrives, 
bo3 = BayesianOptimizer(x3w3, y3w3, acquisition=acquisition3, beta=beta3, noise=noise3, xi=xi3, length_scale_bounds=length_scale_bounds3)
next3 = bo3.propose(n_candidates, lower_bound, upper_bound)
print()
print("The next points to be submitted are: ")
print_next_submission(next3)


*** Loading new inputs for week 3, function 3 ***

The next points to be submitted are: 
0.800000-0.800000-0.800000


## Week 4

In [330]:
x3w4, y3w4 = load_and_append(x3, y3, 4, 3)
#create_and_display_df(x3w4, y3w4, columns3)
xi3=0.015
acquisition3='ei'
n_candidates=20000
lower_bound=0
upper_bound=0.7
noise3=1e-3
scale3=0.01
length_scale_bounds3=(0.05, 2.0) # lets gp adapt smootheness as new data arrives, 
bo3 = BayesianOptimizer(
    x3w4, y3w4
    , acquisition=acquisition3, beta=beta3, noise=noise3, xi=xi3, length_scale_bounds=length_scale_bounds3)
next3 = bo3.propose(n_candidates, lower_bound, upper_bound, scale3)
print()
print("The next points to be submitted are: ")
print_next_submission(next3)


*** Loading new inputs for week 4, function 3 ***

The next points to be submitted are: 
0.403526-0.251754-0.481195


* Last weeks results were 'bad' but good for confirming suspisions.
* I am moving more towards exploitation. In this field, even though I am not knowledgeable, I assume that drastically changing doses does not yield effective results, so concentrating on whats known to be good will be beneficial
* xi has been lowered
* tried to open the bounds slightly, still keeping me away from extremes. However, it is still pointing me to the edges, because the might think that we dont know much about the region. from analysing the points, high x1 and x3s have not produced good results, so i will refine the bounds to be tighter, to truly capitalize on the best observed region so far
* noise as also been increased, for less overconfident spikes, more stable ei behaviour, and smoother predictions
* i adjusted the propose function for ei, instead of generating random uniform candidates, for ei, i would generate random normal number, to exploit the bell curve more

## Week 5

In [339]:
x3w5, y3w5 = load_and_append(x3, y3, 5, 3)
#create_and_display_df(x3w5, y3w5, columns3)
xi3=0.025
acquisition3='ei'
n_candidates=30000
lower_bound=0.15
upper_bound=0.8
noise3=1e-3
scale3=0.02
length_scale_bounds3=(0.05, 2.0) # lets gp adapt smootheness as new data arrives, 
bo3 = BayesianOptimizer(
    x3w5, y3w5
    , acquisition=acquisition3, beta=beta3, noise=noise3, xi=xi3, length_scale_bounds=length_scale_bounds3)
next3 = bo3.propose(n_candidates, lower_bound, upper_bound, scale3)
print()
print("The next points to be submitted are: ")
print_next_submission(next3)


*** Loading new inputs for week 5, function 3 ***

The next points to be submitted are: 
0.395695-0.196177-0.532541


* Last weeks results were not bad
    *  xi raised
    *  lower bound raised
    *  upper bound raised
    *  scale increased

# Function 4

Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.

## Week 1

In [340]:
x4, y4 = load_data(4, 1)

columns4 = ['x1', 'x2', 'x3', 'x4']
#create_and_display_df(x4, y4, columns4)
acquisition4 = "ucb"
beta4 = 1.5
noise4 = 1e-6
xi4 = 0.005

bo4 = BayesianOptimizer(x4, y4, acquisition=acquisition4, beta=beta4, noise=noise4)
next4 = bo4.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next4)


function4/week1

The next points to be submitted are: 
0.960207-0.068080-0.000042-0.131399


## Week 2

In [341]:
x4w2, y4w2 = load_and_append(x4, y4, 2, 4)
#create_and_display_df(x4w2, y4w2, columns4)
#print_shapes(x4w2, y4w2)
acquisition4='ucb'
bo4 = BayesianOptimizer(x4w2, y4w2, acquisition=acquisition4, beta=beta4, noise=noise4, xi=xi4)
next4 = bo4.propose()

print("The next points to be submitted are: ")
print_next_submission(next4)


*** Loading new inputs for week 2, function 4 ***
The next points to be submitted are: 
0.899896-0.001424-0.913922-0.964324


* ucb still better choice. understanding prompt better now

## Week 3

In [342]:
x4w3, y4w3 = load_and_append(x4, y4, 3, 4)
#create_and_display_df(x4w3, y4w3, columns4)
#print_shapes(x4w2, y4w2)
acquisition4='ucb'
beta4=2.0
noise4=1e-4
bo4 = BayesianOptimizer(x4w3, y4w3, acquisition=acquisition4, beta=beta4, noise=noise4, xi=xi4)
next4 = bo4.propose()

print("The next points to be submitted are: ")
print_next_submission(next4)


*** Loading new inputs for week 3, function 4 ***
The next points to be submitted are: 
0.114733-0.414210-0.979896-0.993211


* switching between ucb and ei, to try and see robustness, both give similar regions? I need to understand this better

## Week 4

In [17]:
x4w4, y4w4 = load_and_append(x4, y4, 4, 4)
#create_and_display_df(x4w4, y4w4, columns4)

acquisition4='ei'
beta4=1.7
noise4=1e-4
xi4=0.01
length_scale_bounds4=(0.08, 1.0)
bo4 = BayesianOptimizer(
    x4w4, y4w4
    , acquisition=acquisition4, beta=beta4, noise=noise4, xi=xi4, length_scale_bounds=length_scale_bounds4)
next4 = bo4.propose(20000)

print("The next points to be submitted are: ")
print_next_submission(next4)


*** Loading new inputs for week 4, function 4 ***
The next points to be submitted are: 
0.549933-0.520237-0.414532-0.389236


* switched to ei, after revloutionizing the function after function3, i decided to try and trust it.
* the points given seem to be fairly consistent, hopefully will be good
* length_scale_bounds also added, to avoid tiny fluctuations in mid-range regions, keeping the GP flexible at edges

## Week 5

In [366]:
x4w5, y4w5 = load_and_append(x4, y4, 5, 4)
#create_and_display_df(x4w5, y4w5, columns4)
acquisition4='ei'
beta4=1.7
noise4=1e-4
xi4=0.005
length_scale_bounds4=(0.08, 1.0)
bo4 = BayesianOptimizer(
    x4w5, y4w5
    , acquisition=acquisition4, beta=beta4, noise=noise4, xi=xi4, length_scale_bounds=length_scale_bounds4)
next4 = bo4.propose(20000)

print("The next points to be submitted are: ")
print_next_submission(next4)


*** Loading new inputs for week 5, function 4 ***
The next points to be submitted are: 
0.493095-0.510769-0.417421-0.467318


* update ei to be as originally intended
* xi reduced

# Function 5

You’re tasked with optimising a four-variable black-box function that represents the yield of a chemical process in a factory. The function is typically unimodal, with a single peak where yield is maximised.

Your goal is to find the optimal combination of chemical inputs that delivers the highest possible yield, using systematic exploration and optimisation methods.

## Week 1

In [394]:
x5, y5 = load_data(5, 1)

columns5 = ['x1', 'x2', 'x3', 'x4']
create_and_display_df(x5, y5, columns5)
acquisition5 = "ei"
beta5 = 1.5
noise5 = 1e-6
xi5 = 0.005

bo5 = BayesianOptimizer(x5, y5, acquisition=acquisition5, beta=beta5, noise=noise5, xi=xi5)
next5 = bo5.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next5)


function5/week1
(20, 4)
(20,)


,x1,x2,x3,x4,output
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370525
5,0.683432,0.118663,0.829046,0.567577,78.434389
6,0.553621,0.667350,0.323806,0.814870,57.571537
7,0.352356,0.322242,0.116979,0.473113,109.571876
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223610



The next points to be submitted are: 
0.210347-0.851409-0.902702-0.900652


## Week 2

In [368]:
x5w2, y5w2 = load_and_append(x5, y5, 2, 5)
#create_and_display_df(x5w2, y5w2, columns5)
#print_shapes(x5w2, y5w2)
acquisition5='ei'
xi5 = 0.01 # less exploration

bo5 = BayesianOptimizer(x5w2, y5w2, acquisition=acquisition5, beta=beta5, noise=noise5, xi=xi5, length_scale=0.1)
next5 = bo5.propose()

print("The next points to be submitted are: ")
print_next_submission(next5)


*** Loading new inputs for week 2, function 5 ***
The next points to be submitted are: 
0.235083-0.841164-0.875342-0.907577


* ei, found a high peak, better to capitalize and reduce exploration

## Week 3

In [369]:
x5w3, y5w3 = load_and_append(x5, y5, 3, 5)
#create_and_display_df(x5w3, y5w3, columns5)
acquisition5='ei'
xi5 = 0.001 # less exploration - ok for now

y_mean = np.mean(y5w3)
y_std = np.std(y5w3)
y5w3_scaled = (y5w3 - y_mean) / y_std

bo5 = BayesianOptimizer(x5w3, y5w3, acquisition=acquisition5, beta=beta5, noise=noise5, xi=xi5, length_scale=0.25)
next5 = bo5.propose()

print("The next points to be submitted are: ")
print_next_submission(next5)


*** Loading new inputs for week 3, function 5 ***
The next points to be submitted are: 
0.357382-0.912736-1.000000-0.978217


## Week 4

In [370]:
x5w4, y5w4 = load_and_append(x5, y5, 4, 5)
#create_and_display_df(x5w4, y5w4, columns5)
acquisition5='ei'
xi5 = 0.001
upper_bound5=0.999999
y_mean = np.mean(y5w4)
y_std = np.std(y5w4)
y5w4_scaled = (y5w4 - y_mean) / y_std

bo5 = BayesianOptimizer(
    x5w4, y5w4_scaled
    , acquisition=acquisition5, beta=beta5, noise=noise5, xi=xi5, length_scale=0.25, )
next5 = bo5.propose(20000, upper_bound=upper_bound5)

print("The next points to be submitted are: ")
print_next_submission(next5)


*** Loading new inputs for week 4, function 5 ***
The next points to be submitted are: 
0.391176-0.914401-0.997634-0.958910


* ei again, seemed to be reaching the peak
* scaling is optional, but im choosing to leave it for now
* upped n_candidates to 20000, for good measure (might be the standard from now on)
* massive jump this time, quite happy with this particular function
* had to  lower the upper bound when proposing, it was opting 1 for x3 and x4

## Week 5

In [396]:
x5w5, y5w5 = load_and_append(x5, y5, 5, 5)
create_and_display_df(x5w5, y5w5, columns5)
acquisition5='ei'
xi5 = 0.001
upper_bound5=0.999999
y_mean = np.mean(y5w5)
y_std = np.std(y5w5)
y5w5_scaled = (y5w5 - y_mean) / y_std

bo5 = BayesianOptimizer(
    x5w5, y5w5
    , acquisition=acquisition5, beta=beta5, noise=noise5, xi=xi5, length_scale=0.25, )
next5 = bo5.propose(20000, upper_bound=upper_bound5)

print("The next points to be submitted are: ")
print_next_submission(next5)


*** Loading new inputs for week 5, function 5 ***
(24, 4)
(24,)


,x1,x2,x3,x4,output
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370525
5,0.683432,0.118663,0.829046,0.567577,78.434389
6,0.553621,0.667350,0.323806,0.814870,57.571537
7,0.352356,0.322242,0.116979,0.473113,109.571876
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223610


The next points to be submitted are: 
0.424918-0.932187-0.999999-0.999999


* ei again, output is increasing by the thousands, very happy with this
* tiny xi, heavy exploitation
* length scale @ 0.25 keeps predictions smooth, preventing wild jumps in less sampled regions

# Function 6

You’re optimising a cake recipe using a black-box function with five ingredient inputs, for example flour, sugar, eggs, butter and milk. Each recipe is evaluated with a combined score based on flavour, consistency, calories, waste and cost, where each factor contributes negative points as judged by an expert taster. This means the total score is negative by design. 

To frame this as a maximisation problem, your goal is to bring that score as close to zero as possible or, equivalently, to maximise the negative of the total sum.

## Week 1

In [397]:
x6, y6 = load_data(6, 1)

columns6 = ['flour', 'sugar', 'eggs', 'butter', 'milk']
#create_and_display_df(x6, y6, columns6, 'score')
acquisition6 = "ei"
beta6 = 1.96
noise6 = 1e-6
xi6 = 0.005
length_scale6 = 0.1

bo6 = BayesianOptimizer(x6, y6, acquisition=acquisition6, beta=beta6, noise=noise6)
next6 = bo6.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next6)


function6/week1

The next points to be submitted are: 
0.929540-0.135436-0.991054-0.000000-0.078374


## Week 2

In [398]:
x6w2, y6w2 = load_and_append(x6, y6, 2, 6)
#create_and_display_df(x6w2, y6w2, columns6)
#print_shapes(x6w2, y6w2)
acquisition6='ei'
bo6 = BayesianOptimizer(x6w2, y6w2, acquisition=acquisition6, beta=beta6, noise=noise6, xi=xi6, length_scale=length_scale6)
next6 = bo6.propose()

print("The next points to be submitted are: ")
print_next_submission(next6)


*** Loading new inputs for week 2, function 6 ***
The next points to be submitted are: 
0.002316-0.896161-0.017548-0.640417-0.000000


easy, i think its a matter of fine tuning hyperparameters and xi for exploration

## Week 3

In [399]:
x6w3, y6w3 = load_and_append(x6, y6, 3, 6)
#create_and_display_df(x6w3, y6w3, columns6)

noise6 = 1e-6
xi6 = 0.005
length_scale6 = 0.2
acquisition6='ei'
bo6 = BayesianOptimizer(x6w3, y6w3, acquisition=acquisition6, beta=beta6, noise=noise6, xi=xi6, length_scale=length_scale6)
next6 = bo6.propose()

print("The next points to be submitted are: ")
print_next_submission(next6)


*** Loading new inputs for week 3, function 6 ***
The next points to be submitted are: 
0.370344-0.000000-1.000000-0.000094-0.064188


* easy, i think its a matter of fine tuning hyperparameters and xi for exploration
* length_scale6=0.2 to let the GP be slightly smoother

## Week 4

In [401]:
x6w4, y6w4 = load_and_append(x6, y6, 4, 6)
#create_and_display_df(x6w4, y6w4, columns6)

noise6 = 1e-4
xi6 = 0.1
length_scale6 = 0.01
acquisition6='ei'
bo6 = BayesianOptimizer(
    x6w4, y6w4
    , acquisition=acquisition6, beta=beta6, noise=noise6, xi=xi6, length_scale=length_scale6)
next6 = bo6.propose(20000)

print("The next points to be submitted are: ")
print_next_submission(next6)


*** Loading new inputs for week 4, function 6 ***
The next points to be submitted are: 
0.480797-0.310986-0.518895-0.949960-0.088197


* increased xi and noise, making ei less greedy. using last week params gave horrible edge cases. not quite sure why, need to look into this (more realistic recipe combinations as opposed to 0,0,1,0,1)
* prevents proposals at the edge of the input space
* decreased length_scale. GP more sensitive to local variations in the data. fine tuning around promising areas rather than smoothing too much

## Week 5

In [441]:
x6w5, y6w5 = load_and_append(x6, y6, 5, 6)
#create_and_display_df(x6w5, y6w5, columns6)

noise6 = 1e-6
xi6 = 0.01
lower_bound6=0.005000
upper_bound6=0.999999
length_scale6 = 0.25
acquisition6='ei'
bo6 = BayesianOptimizer(
    x6w5, y6w5
    , acquisition=acquisition6, beta=beta6, noise=noise6, xi=xi6, length_scale=length_scale6)
next6 = bo6.propose(20000, lower_bound=lower_bound6, upper_bound=upper_bound6)

print("The next points to be submitted are: ")
print_next_submission(next6)


*** Loading new inputs for week 5, function 6 ***
The next points to be submitted are: 
0.563063-0.055132-0.757525-0.834489-0.039061


* lowered noise
* lowered xi
* raised length scale
* added lower bound to avoid edges

# Function 7

You’re tasked with optimising an ML model by tuning six hyperparameters, for example learning rate, regularisation strength or number of hidden layers. The function you’re maximising is the model’s performance score (such as accuracy or F1), but since the relationship between inputs and output isn’t known, it’s treated as a black-box function. 

Because this is a commonly used model, you might benefit from researching best practices or literature to guide your initial search space. Your goal is to find the combination of hyperparameters that yields the highest possible performance.

## Week 1

In [442]:
x7, y7 = load_data(7, 1)

columns7 = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
#create_and_display_df(x7, y7, columns7, 'accuracy')
acquisition7 = "ucb"
beta7 = 1.96
noise7 = 1e-6
xi7 = 0.005
length_scale7 = 0.1

bo7 = BayesianOptimizer(x7, y7, acquisition=acquisition7, beta=beta7, noise=noise7)
next7 = bo7.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next7)


function7/week1

The next points to be submitted are: 
0.214520-0.423480-0.340972-0.348082-0.395394-0.644118


## Week 2

In [443]:
x7w2, y7w2 = load_and_append(x7, y7, 2, 7)
#create_and_display_df(x7w2, y7w2, columns7)
#print_shapes(x7w2, y7w2)
acquisition7='ucb'
noise7=1e-4
bo7 = BayesianOptimizer(x7w2, y7w2, acquisition=acquisition7, beta=beta7, noise=noise7, xi=xi7, length_scale=length_scale7)
next7 = bo7.propose()

print("The next points to be submitted are: ")
print_next_submission(next7)


*** Loading new inputs for week 2, function 7 ***
The next points to be submitted are: 
0.045185-0.174454-0.299620-0.075273-0.160976-0.980098


easy, i think its a matter of fine tuning beta for exploration

## Week 3

In [444]:
x7w3, y7w3 = load_and_append(x7, y7, 3, 7)
#create_and_display_df(x7w3, y7w3, columns7)
acquisition7='ucb'

beta7 = 2.5
noise7 = 1e-6
xi7 = 0.005
length_scale7 = 0.3
n_candidates=20000
length_scale_bounds7=(1e-2, 1)
bo7 = BayesianOptimizer(x7w3, y7w3, acquisition=acquisition7, beta=beta7, noise=noise7, xi=xi7, length_scale=length_scale7, length_scale_bounds=length_scale_bounds7)
next7 = bo7.propose(n_candidates)

print("The next points to be submitted are: ")
print_next_submission(next7)


*** Loading new inputs for week 3, function 7 ***
The next points to be submitted are: 
0.005208-0.736004-0.009731-0.112812-0.393411-0.163801


* beta increased, noise reduced, candidates increased dramatically

## Week 4

In [445]:
x7w4, y7w4 = load_and_append(x7, y7, 4, 7)
#create_and_display_df(x7w4, y7w4, columns7)
acquisition7='ei'
def lower_bound_15_percent(x):
    """
    Returns a number 15% smaller than the input x.
    """
    return x * (1 - 0.15)

beta7 = 2.5
noise7 = 1e-8
xi7 = 0.005
length_scale7 = 0.005
n_candidates=20000
length_scale_bounds7=(1e-2, 1)
lower_bound7 = x7w4[:,0].min()
print("Lowest x1:", lower_bound7)
lower_bound7 = lower_bound_15_percent(lower_bound7)

bo7 = BayesianOptimizer(
    x7w4, y7w4
    , acquisition=acquisition7, beta=beta7, noise=noise7, xi=xi7, length_scale=length_scale7, length_scale_bounds=length_scale_bounds7)
next7 = bo7.propose(n_candidates, lower_bound=lower_bound7 )

print("The next points to be submitted are: ")
print_next_submission(next7)




*** Loading new inputs for week 4, function 7 ***
Lowest x1: 0.030789
The next points to be submitted are: 
0.026171-0.441917-0.236053-0.220913-0.394948-0.717695


* switched to ei, want to capitalize on the known optimum
* had to tweek the lower bound. x1 kept giving me 0, so i decided to fix it to the lowest x1 recorded +- 15%

## Week 5

In [507]:
x7w5, y7w5 = load_and_append(x7, y7, 5, 7)
#create_and_display_df(x7w5, y7w5, columns7)
acquisition7='ei'
def lower_bound_15_percent(x):
    """
    Returns a number 15% smaller than the input x.
    """
    return x * (1 - 0.15)

beta7 = 2.5
noise7 = 1e-8
xi7 = 0.01
length_scale7 = 0.01
n_candidates=20000
length_scale_bounds7=(1e-2, 1)
lower_bound7 = x7w5[:,0].min()
print("Lowest x1:", lower_bound7)
lower_bound7 = lower_bound_15_percent(lower_bound7)
lower_bound7 = 0

bo7 = BayesianOptimizer(
    x7w5, y7w5
    , acquisition=acquisition7, beta=beta7, noise=noise7, xi=xi7, length_scale=length_scale7, length_scale_bounds=length_scale_bounds7)
next7 = bo7.propose(n_candidates, lower_bound=lower_bound7 )

print("The next points to be submitted are: ")
print_next_submission(next7)




*** Loading new inputs for week 5, function 7 ***
Lowest x1: 0.026171
The next points to be submitted are: 
0.016636-0.450686-0.319713-0.227405-0.452364-0.785255


* kept ei, but tweaked a few params
* lower bound has been reset to 0
* xi is slightly higher to encourage 0.005 more exploration lol
* length_scale has been increased for a slightly smoother model

# Function 8

You’re optimising an eight-dimensional black-box function, where each of the eight input parameters affects the output, but the internal mechanics are unknown.

Your objective is to find the parameter combination that maximises the function’s output, such as performance, efficiency or validation accuracy. Because the function is high-dimensional and likely complex, global optimisation is hard, so identifying strong local maxima is often a practical strategy.

For example, imagine you’re tuning an ML model with eight hyperparameters: learning rate, batch size, number of layers, dropout rate, regularisation strength, activation function (numerically encoded), optimiser type (encoded) and initial weight range. Each input set returns a single validation accuracy score between 0 and 1. Your goal is to maximise this score.

## Week 1

In [508]:
x8, y8 = load_data(8, 1)

columns8 = ['learning_rate', 'batch_size', 'num_layers', 'dropout_rate', 'regularisation_strength', 'activation_function', 'optimiser_type', 'initial_weight_range']
#create_and_display_df(x8, y8, columns8, 'validation_accuracy')
acquisition8 = "ucb"
beta8 = 3
noise8 = 1e-6
xi8 = 0.005
length_scale8 = 0.7

bo8 = BayesianOptimizer(x8, y8, acquisition=acquisition8, beta=beta8, noise=noise8)
next8 = bo8.propose()
print()
print("The next points to be submitted are: ")
print_next_submission(next8)


function8/week1

The next points to be submitted are: 
0.560970-0.316734-0.881709-0.347299-0.115233-0.559504-0.310690-0.675898


## Week 2

In [509]:
x8w2, y8w2 = load_and_append(x8, y8, 2, 8)
#create_and_display_df(x8w2, y8w2, columns8)
#print_shapes(x8w2, y8w2)
acquisition8='ucb'
noise8=1e-4
bo8 = BayesianOptimizer(x8w2, y8w2, acquisition=acquisition8, beta=beta8, noise=noise8, xi=xi8, length_scale=length_scale8)
next8 = bo8.propose()

print("The next points to be submitted are: ")
print_next_submission(next8)


*** Loading new inputs for week 2, function 8 ***
The next points to be submitted are: 
0.091834-0.192544-0.241299-0.124382-0.360966-0.465524-0.574485-0.388319


easy, i think its a matter of fine tuning beta for exploration

## Week 3

In [510]:
x8w3, y8w3 = load_and_append(x8, y8, 3, 8)
#create_and_display_df(x8w3, y8w3, columns8)
acquisition8='ei'
noise8=1e-6
beta8 = 2.0
xi8 = 0.01
length_scale8 = 0.3
length_scale_bounds8=(0.1, 2.0)
n_candidates=20000

bo8 = BayesianOptimizer(x8w3, y8w3, acquisition=acquisition8, beta=beta8, noise=noise8, xi=xi8, length_scale=length_scale8,length_scale_bounds=length_scale_bounds8)
next8 = bo8.propose(n_candidates)

print("The next points to be submitted are: ")
print_next_submission(next8)


*** Loading new inputs for week 3, function 8 ***
The next points to be submitted are: 
0.092075-0.216847-0.147519-0.246869-0.552303-0.590576-0.234861-0.462187


* switched to ei
* lower beta value for ucb, to balances more exporing while capitalizing on promising regions, not as aggressive as week 2
* lower length scale, and added length scale bounds, prevents underfitting too much
* reduced noise
* 0.141077-0.258349-0.159880-0.254071-0.539345-0.660506-0.326839-0.704651 as sample points with ucb

## Week 4

In [511]:
x8w4, y8w4 = load_and_append(x8, y8, 4, 8)
#create_and_display_df(x8w4, y8w4, columns8)
acquisition8='ei'
noise8=1e-6
beta8 = 2.0
xi8 = 0.01
length_scale8 = 0.3
length_scale_bounds8=(0.1, 2.0)
n_candidates=20000

bo8 = BayesianOptimizer(
    x8w4, y8w4
    , acquisition=acquisition8, beta=beta8, noise=noise8, xi=xi8, length_scale=length_scale8,length_scale_bounds=length_scale_bounds8)
next8 = bo8.propose(n_candidates)

print("The next points to be submitted are: ")
print_next_submission(next8)


*** Loading new inputs for week 4, function 8 ***
The next points to be submitted are: 
0.142302-0.350007-0.094652-0.214567-0.501315-0.564332-0.253200-0.484871


* switched to ei
* committing to ei, the points seem consistent, and outputs are clearly going in the right direction
* next week i will focus on exploration, exploring another local maxima. im interested in point index 14, will try to focus in on that, find another maxima
* all hyperparams are the same as last week, want to see the direction for this week

## Week 5

In [566]:
x8w5, y8w5 = load_and_append(x8, y8, 5, 8)
#create_and_display_df(x8w5, y8w5, columns8)
acquisition8='ei'
noise8=1e-6
beta8 = 2.0
xi8 = 0.02
length_scale8 = 0.25
length_scale_bounds8=(0.1, 1.6)
n_candidates=20000

bo8 = BayesianOptimizer(
    x8w5, y8w5
    , acquisition=acquisition8, beta=beta8, noise=noise8, xi=xi8, length_scale=length_scale8,length_scale_bounds=length_scale_bounds8)
next8 = bo8.propose(n_candidates)

print("The next points to be submitted are: ")
print_next_submission(next8)


*** Loading new inputs for week 5, function 8 ***
The next points to be submitted are: 
0.129283-0.111173-0.266707-0.161535-0.651723-0.504693-0.216420-0.562856


* committed to ei
* xi slightly increased, preserve exploitation, but allows some wiggle room for exploring
* length scale reduced, can capture more sudden changes in high dimensional function